In [13]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path



In [14]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #find all files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)


Found 1 PDF files to process

Processing: anudeep_manda_openai_coverletter.pdf
  ✓ Loaded 1 pages

Total documents loaded: 1


In [6]:
all_pdf_documents

[Document(metadata={'producer': 'macOS Version 15.6 (Build 24G84) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20260116012304Z00'00'", 'title': 'anudeep_dropbox_coverletter', 'author': 'Anudeep Manda', 'moddate': "D:20260116012304Z00'00'", 'source': '../data/pdf/anudeep_manda_openai_coverletter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'anudeep_manda_openai_coverletter.pdf', 'file_type': 'pdf'}, page_content='Anudeep Manda manda1@purdue.edu | (925) 683-7968 | linkedin.com/in/anudeepmanda Dear OpenAI Applied Engineering Team, I am writing to express my strong interest in the Software Engineering Intern (Emerging Talent) role on the Applied Engineering team at OpenAI. I am a Computer Engineering and Applied Mathematics student at Purdue University, and I am deeply motivated by OpenAI’s mission to responsibly deploy AI systems that are useful, reliable, and accessible to millions of users worldwide. OpenAI’s emphasis on learning from real-world depl

In [15]:
def split_documents(document, chunk_size =1000, chunk_overlap = 200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len, 
        separators=  ["\n\n", "\n", " ", '']
    )
    split_docs = text_splitter.split_documents(document)
    print(f"Split {len(document)} documents into {len(split_docs)} chunks")
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [17]:
chunks = split_documents(all_pdf_documents)
chunks

Split 1 documents into 4 chunks

Example chunk:
Content: Anudeep Manda manda1@purdue.edu | (925) 683-7968 | linkedin.com/in/anudeepmanda Dear OpenAI Applied Engineering Team, I am writing to express my strong interest in the Software Engineering Intern (Eme...
Metadata: {'producer': 'macOS Version 15.6 (Build 24G84) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20260116012304Z00'00'", 'title': 'anudeep_dropbox_coverletter', 'author': 'Anudeep Manda', 'moddate': "D:20260116012304Z00'00'", 'source': '../data/pdf/anudeep_manda_openai_coverletter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'anudeep_manda_openai_coverletter.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 15.6 (Build 24G84) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20260116012304Z00'00'", 'title': 'anudeep_dropbox_coverletter', 'author': 'Anudeep Manda', 'moddate': "D:20260116012304Z00'00'", 'source': '../data/pdf/anudeep_manda_openai_coverletter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'anudeep_manda_openai_coverletter.pdf', 'file_type': 'pdf'}, page_content='Anudeep Manda manda1@purdue.edu | (925) 683-7968 | linkedin.com/in/anudeepmanda Dear OpenAI Applied Engineering Team, I am writing to express my strong interest in the Software Engineering Intern (Emerging Talent) role on the Applied Engineering team at OpenAI. I am a Computer Engineering and Applied Mathematics student at Purdue University, and I am deeply motivated by OpenAI’s mission to responsibly deploy AI systems that are useful, reliable, and accessible to millions of users worldwide. OpenAI’s emphasis on learning from real-world depl

In [ ]:
# Embedding and Vector Store

In [18]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [19]:
class EmbeddingManager:
    #handles domcument embedding gneration using SentenceTransfomer

    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        '''
        Initialie the embedding manager
        model_name -> HuggingFace model name for sentences embeddings

        '''
        self.model_name = model_name
        self.model = None
        self._load_model() #loads model 

    def _load_model(self): #protected funciton 
        try:
            print(f"Loading embdding model : {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded succesfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}") # every text converted into dimensions
        except Exception as e:
            print(f"Error loading model {self.model}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]):
        if not self.model: 
            raise ValueError("Model not loaded")
        print(f"Generate Embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar= True)
        print(f"Generate embeddings with shape: {embeddings.shape}")
        return embeddings


##initializze the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embdding model : all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2021.75it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded succesfully. Embedding dimension: 384


In [ ]:
#vector store

In [36]:
import numpy as np
import os

class VectorStore:
    def __init__(self, collection_name: str = 'pdf_documents', persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try: 
            #create persistent chromadb client 
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #create or get collection 
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description": "PDF document embeddings for RAG"}

            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e: 
            print(f"Error intializing vector store: {e}")
            raise
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
            #add documents and their embeddings to the vector store 
            #args: documents: list of langchain documents, embeddings their corresponding embeddings for the documents

            if len(documents) != len(embeddings):
                raise ValueError("Number of documents must match with the number of embeddings")
            
            print(f"Addings {len(documents)} documents to the vector store")
            

            #data preperation for ChromaDB 
            ids= []
            metadata_list = []
            documents_text = []
            embeddings_list = []

            for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
                #genereate unique id
                doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
                ids.append(doc_id)

                #prepare metadata
                metadata = dict(doc.metadata)
                metadata['doc_index'] = i 
                metadata['content_length'] = len(doc.page_content)
                metadata_list.append(metadata)
                #documents content
                documents_text.append(doc.page_content)
                #embedding
                embeddings_list.append(embedding.tolist())


            #add to collection 
            try:
                self.collection.add(
                    ids = ids, 
                    embeddings = embeddings, 
                    metadatas = metadata_list,
                    documents = documents_text

                )
                print(f"Successfully added {len(documents)} chunks to the vector store")
                print(f"Total chunks in collection: {self.collection.count()}")


            except Exception as e: 
                print(f"Error adding chunks to vector store: {e}")
                raise


vectorstore = VectorStore()
vectorstore




Vector store initialized. Collection: pdf_documents
Existing documents in collection: 8


In [37]:
chunks

[Document(metadata={'producer': 'macOS Version 15.6 (Build 24G84) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20260116012304Z00'00'", 'title': 'anudeep_dropbox_coverletter', 'author': 'Anudeep Manda', 'moddate': "D:20260116012304Z00'00'", 'source': '../data/pdf/anudeep_manda_openai_coverletter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'anudeep_manda_openai_coverletter.pdf', 'file_type': 'pdf'}, page_content='Anudeep Manda manda1@purdue.edu | (925) 683-7968 | linkedin.com/in/anudeepmanda Dear OpenAI Applied Engineering Team, I am writing to express my strong interest in the Software Engineering Intern (Emerging Talent) role on the Applied Engineering team at OpenAI. I am a Computer Engineering and Applied Mathematics student at Purdue University, and I am deeply motivated by OpenAI’s mission to responsibly deploy AI systems that are useful, reliable, and accessible to millions of users worldwide. OpenAI’s emphasis on learning from real-world depl

In [ ]:
#convert chunks texts in embeddings

texts = [doc.page_content for doc in chunks]
texts

#generate the embeddings

embeddings = embedding_manager.generate_embeddings(texts)

#store in the vector database
vectorstore.add_documents(chunks, embeddings)

# how to remove duplicates?


Generate Embeddings for 4 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.55it/s]

Generate embeddings with shape: (4, 384)
Addings 4 documents to the vector store
Successfully added 4 chunks to the vector store
Total chunks in collection: 12
